# 实践 17.3  稳定表观等位基因与环境诱导甲基化的跨代选择价值

《统计遗传与智能育种学》第十七章 · 电子资源 `17c_epigenetic_breeding.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：17c_epigenetic_breeding.ipynb。实践构造720个亲代—后代配对家系，同时模拟高保持率表观等位基因与每代重新响应环境的甲基化标记。120个独立校准家系只用于估计每个标记的跨代相关，360个家系只用于第一代表型模型训练，剩余240个候选家系用于第二代选择响应评价；稳定性筛选仅使用可观测的跨代相关，不读取模拟真效应。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果见下方代码输出；必须报告稳定/环境诱导标记的跨代相关、通过稳定性阈值的特征数、第二代候选均值及两种选择策略的响应。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1703；720个亲代—后代家系；40个稳定表观等位基因+40个环境诱导标记；120校准/360训练/240候选；选择60个家系；无外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 1703
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)

n, k = 720, 40
# 相同家系DNA背景在亲代和后代高度相关，但不是完全相同。
G1 = rng.normal(size=(n,20))
G2 = 0.90*G1 + np.sqrt(1-0.90**2)*rng.normal(size=G1.shape)

stable1 = rng.binomial(1, 0.45, size=(n,k))
keep = rng.random((n,k)) < 0.90
stable2 = np.where(keep, stable1, rng.binomial(1,0.45,size=(n,k)))

env1 = rng.integers(0,2,n)
env2 = rng.integers(0,2,n)
induced1 = 0.9*env1[:,None] + rng.normal(0,0.55,size=(n,k))
induced2 = 0.9*env2[:,None] + rng.normal(0,0.55,size=(n,k))

beta_epi = rng.normal(0,0.12,k)
y1 = stable1@beta_epi + 0.45*induced1[:,:5].mean(axis=1) + 0.35*G1[:,0] + rng.normal(0,1,n)
y2 = stable2@beta_epi + 0.45*induced2[:,:5].mean(axis=1) + 0.35*G2[:,0] + rng.normal(0,1,n)
print('parent_offspring_pairs/features', n, 2*k)

# 120个校准家系只用于估计每个表观标记的跨代稳定性；360个家系训练第一代表型模型；其余240个用于选择响应评价。
cal = np.arange(120)
train = np.arange(120,480)
cand = np.arange(480,n)

X1 = np.hstack([stable1, induced1])
X2 = np.hstack([stable2, induced2])

def marker_corr(a,b,idx):
    out=[]
    for j in range(a.shape[1]):
        r=np.corrcoef(a[idx,j],b[idx,j])[0,1]
        out.append(0.0 if not np.isfinite(r) else r)
    return np.array(out)

r_stable = marker_corr(stable1,stable2,cal)
r_induced = marker_corr(induced1,induced2,cal)
stability = np.r_[r_stable,r_induced]
keep_features = stability >= 0.50
print('mean_crossgen_r stable/induced', round(r_stable.mean(),3), round(r_induced.mean(),3))
print('features_passing_r_ge_0.50', int(keep_features.sum()), '/', 2*k)

def fit_score(X):
    model=make_pipeline(StandardScaler(), Ridge(alpha=40.0))
    model.fit(X[train], y1[train])
    return model.predict(X[cand])

score_naive = fit_score(X1)
score_stable = fit_score(X1[:,keep_features])
n_select = 60
sel_naive = cand[np.argsort(score_naive)[-n_select:]]
sel_stable = cand[np.argsort(score_stable)[-n_select:]]
baseline = y2[cand].mean()
print('generation2_candidate_mean', round(float(baseline),3))
print('generation2_selected_mean naive_all_epi', round(float(y2[sel_naive].mean()),3),
      'response', round(float(y2[sel_naive].mean()-baseline),3))
print('generation2_selected_mean stability_filtered', round(float(y2[sel_stable].mean()),3),
      'response', round(float(y2[sel_stable].mean()-baseline),3))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1703
parent_offspring_pairs/features 720 80
mean_crossgen_r stable/induced 0.898 -0.038
features_passing_r_ge_0.50 40 / 80
generation2_candidate_mean -0.196
generation2_selected_mean naive_all_epi -0.018 response 0.177
generation2_selected_mean stability_filtered 0.052 response 0.248


结果解释：稳定性阈值完全由独立校准家系的跨代观测估计，性状效应完全由第一代训练家系估计，两者都不使用模拟真效应。本例选择响应指入选家系后代表型均值相对候选群体均值的差；单次模拟中某种策略具有更高响应不能外推为普遍结论，重点在于能够追溯各项证据的来源。

可修改练习：将表观等位基因保持概率从0.90改为0.70或0.98，或把跨代相关阈值0.50改为0.30/0.70，观察保留特征数与第二代选择响应。